# Tratamento dos dados e geração das bases

Use este notebook para:

1. localizar os arquivos de origem;
2. montar a base analítica por aluno e ano;
3. gerar a base temporal usada na modelagem;
4. treinar o modelo e salvar os artefatos.

In [1]:


import sys
from pathlib import Path


DIRETORIO_NOTEBOOKS = Path.cwd()
if not (DIRETORIO_NOTEBOOKS / "ambiente_notebook.py").exists():
    candidato = DIRETORIO_NOTEBOOKS / "notebooks"
    if candidato.exists():
        DIRETORIO_NOTEBOOKS = candidato
if str(DIRETORIO_NOTEBOOKS) not in sys.path:
    sys.path.insert(0, str(DIRETORIO_NOTEBOOKS))

from ambiente_notebook import adicionar_raiz_no_syspath, caminho_relativo_projeto

DIRETORIO_RAIZ = adicionar_raiz_no_syspath()

import pandas as pd
from IPython.display import display

from src.passos_magicos_dt.analysis import build_analytics
from src.passos_magicos_dt.data import prepare_datasets, resolve_legacy_csv_path
from src.passos_magicos_dt.materials import write_painel_payload, write_qna_report
from src.passos_magicos_dt.modeling import save_training_artifacts, train_temporal_model

## 1. Conferir as fontes

In [2]:
diretorio_raiz = DIRETORIO_RAIZ
pacote_dados = prepare_datasets(root=diretorio_raiz)
arquivo_legado = resolve_legacy_csv_path(root=diretorio_raiz)

quadro_caminhos = pd.DataFrame(
    [
        {"fonte": "Excel oficial", "caminho": caminho_relativo_projeto(pacote_dados.caminho_excel)},
        {"fonte": "CSV legado", "caminho": caminho_relativo_projeto(arquivo_legado)},
    ]
)
quadro_caminhos

,fonte,caminho
0,Excel oficial,data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx
1,CSV legado,archive/legacy/data/PEDE_PASSOS_DATASET_FIAP.csv


## 2. Conferir a base analítica

In [3]:
resumo_base = pd.DataFrame(
    {
        "registros": [len(pacote_dados.base_analitica)],
        "alunos_unicos": [pacote_dados.base_analitica["ra"].nunique()],
        "anos": [", ".join(map(str, sorted(pacote_dados.base_analitica["ano_referencia"].unique())))],
    }
)
display(resumo_base)
pacote_dados.base_analitica.head()

,registros,alunos_unicos,anos
0,3030,1661,"2022, 2023, 2024"


,ra,nome_exibicao,ano_referencia,origem_aba,fase,turma,ciclo_programa,pedra_atual,genero,idade,...,ipv,ian,categoria_ian,risco_atual,mat,por,ing,media_academica,media_comportamental,defasagem
0,RA-1,Aluno-1,2022,PEDE2022,7,A,Quartzo,Quartzo,Feminino,19.0,...,7.278,5.0,Defasagem moderada,1.0,2.7,3.5,6.0,4.066667,6.000000,-1
1,RA-10,Aluno-10,2022,PEDE2022,7,A,Quartzo,Quartzo,Feminino,18.0,...,7.056,5.0,Defasagem moderada,1.0,3.3,2.6,6.4,4.100000,6.166667,-1
2,RA-100,Aluno-100,2022,PEDE2022,4,A,Ametista,Ametista,Feminino,13.0,...,7.250,10.0,Adequado,0.0,7.0,7.8,8.1,7.633333,7.200000,1
3,RA-101,Aluno-101,2022,PEDE2022,4,A,Ametista,Ametista,Feminino,15.0,...,6.833,5.0,Defasagem moderada,1.0,7.2,6.5,9.2,7.633333,7.900000,-1
4,RA-102,Aluno-102,2022,PEDE2022,4,A,Agata,Agata,Masculino,14.0,...,6.000,10.0,Adequado,0.0,4.7,6.8,5.5,5.666667,7.100000,0


In [4]:
percentual_ausencias = (
    pacote_dados.base_analitica.isna()
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .rename("pct_missing")
    .reset_index()
    .rename(columns={"index": "coluna"})
)
percentual_ausencias

,coluna,pct_missing
0,cf,0.716172
1,ct,0.716172
2,cg,0.716172
3,ano_nasc,0.716172
4,inde_24,0.652145
5,ing,0.639934
6,delta_inde,0.588119
7,inde_anterior,0.574257
8,inde_23,0.465017
9,inde_22,0.362376


## 3. Conferir a base de pares

In [5]:
resumo_pares = pd.DataFrame(
    {
        "registros": [len(pacote_dados.base_pares)],
        "anos_referencia": [", ".join(map(str, sorted(pacote_dados.base_pares["ano_referencia"].unique())))],
        "taxa_risco_proximo_ano": [pacote_dados.base_pares["risco_proximo_ano"].mean()],
    }
)
display(resumo_pares)
pacote_dados.base_pares[["ra", "ano_referencia", "ano_alvo", "risco_atual", "risco_proximo_ano", "delta_inde"]].head()

,registros,anos_referencia,taxa_risco_proximo_ano
0,1365,"2022, 2023",0.493773


,ra,ano_referencia,ano_alvo,risco_atual,risco_proximo_ano,delta_inde
0,RA-1,2022,2023,1.0,0,NaN
1,RA-1,2023,2024,0.0,0,NaN
2,RA-1000,2023,2024,0.0,0,NaN
3,RA-1001,2023,2024,1.0,1,NaN
4,RA-1002,2023,2024,1.0,1,NaN


## 4. Treinar e salvar os artefatos

In [6]:
artefatos_treinamento = train_temporal_model(pacote_dados.base_pares)
save_training_artifacts(artefatos_treinamento)
artefatos_treinamento.metrics_holdout

,model,threshold,accuracy,precision,recall,f1,f2,roc_auc,pr_auc,true_negative,false_positive,false_negative,true_positive
2,random_forest,0.20,0.402614,0.402614,1.000000,0.574091,0.771157,0.646004,0.533167,0.0,457.0,0.0,308.0
0,baseline_regra_negocio,0.15,0.454902,0.413078,0.840909,0.554011,0.696611,0.661030,0.624314,89.0,368.0,49.0,259.0
3,xgboost,0.18,0.462745,0.413445,0.798701,0.544850,0.673235,0.516688,0.397764,108.0,349.0,62.0,246.0
1,regressao_logistica,0.17,0.596078,0.498615,0.584416,0.538117,0.564972,0.602354,0.465214,276.0,181.0,128.0,180.0


In [7]:
artefatos_analiticos = build_analytics(
    pacote_dados.base_analitica,
    pacote_dados.base_pares,
    artefatos_treinamento,
)
write_qna_report(artefatos_analiticos)
write_painel_payload(artefatos_analiticos)
artefatos_treinamento.feature_importance.head(12)

,feature,importance
0,cat__fase_Alfa,3.254201
1,cat__fase_Fase 8,2.757846
2,num__inde_atual,2.501632
3,cat__fase_7,1.958488
4,cat__fase_0,1.637492
5,cat__fase_Fase 7,1.383210
6,cat__ciclo_programa_Nao informado,1.227234
7,cat__fase_6,1.096110
8,cat__fase_Fase 6,1.095792
9,cat__fase_Fase 2,0.925045


## 5. Saídas geradas

Ao final da execução, o projeto atualiza `data/processed/`, `artifacts/model/` e `artifacts/analytics/`.